# 

## Reproduce results for SMARTBind benchmark on RNAmigos1 dataset
This is a performance reproduce script for benchmarking SMARTBind model on RNAmigos1 dataset (both SMARTBind and SMARTBind-noAug). The script contains following information:

1. The script can be reproduced with the pretrained weight downloaded from [here](https://drive.google.com/drive/folders/19uRxv4bYPJ9oCXl0sMNVpzOy663ovQ74?usp=sharing), the no-augmented version can be downloaded from [here](https://drive.google.com/drive/folders/1uUD4-K3pnf6rqoxm3dvCiAQgCe9NSEuP?usp=sharing), and RNAmigos1 10-fold cross-validation dataset downloaded from [here](https://drive.google.com/file/d/1iQBwmtlKUzaxSdl58oEeruMSskoJcXzy/view?usp=sharing), though the reproduce results are already provided below.
3. Training performance on RNAmigos1 dataset taking the PDB ligand in the dataset as decoys, reproduce the results by inference model weights on test set.
4. Evaluation on DecoyFinder datasets (taking ligand-specific decoys found by Decoyfinder from ZINC database).

## Evaluate RNAmigos1 dataset with PDB decoys

In [18]:
import sys
sys.path.append('../../../..')
import os
from smartbind.preprocess import build_val_test_set
from smartbind.dataloader import RLDataLoader, RLDataset
from smartbind import load_smartbind_models
from smartbind.preprocess import convert_smiles_to_pf2

import tqdm
from smartbind.model.margin import sigmoid_cosine_distance_test
import torch
import numpy as np

In [ ]:
current_fold_num = 1
current_path = '../../../../training_data/rnamigos_10fd_with_decoys.pkl'
topk_decoy = 100
batch_size = 24
num_workers = 2
decoy_num = 48
replace_res_type = ['R', 'Y', 'K', 'M', 'S', 'W', 'B', 'D', 'H', 'V', 'N', '-']
device = 'cpu'

In [20]:
full_rank_percentile_list = []
for current_fold_num in range(1, 11):
    print(f'\nProcessing fold {current_fold_num}')
    val_rna_seq_list, val_match_smol_list, val_rna_name_list, val_match_smol_name_list, val_binding_index_list, \
        val_match_smol_fp_list, train_rna_seq_list, train_match_smol_list, train_rna_name_list, \
        train_match_smol_name_list, train_binding_index_list, train_match_smol_fp_list = (
                                        build_val_test_set(
                                            fold_num=current_fold_num-1,
                                            dict_path=current_path,
                                            topk_decoy=topk_decoy))
    val_dataset = RLDataset(rna_sequences=val_rna_seq_list,
                                rna_sequences_names=val_rna_name_list,
                                match_smols=val_match_smol_fp_list,
                                match_smols_names=val_match_smol_name_list,
                                non_binding_index_list=val_binding_index_list,
                                rna_smol_map_path='../../../../smartbind/rnamigos1_rna_smol_map.json',
                                is_val=True)
    val_dataloader = RLDataLoader(dataset=val_dataset,
                                    batch_size=batch_size,
                                    num_workers=num_workers,
                                    if_shuffle=False)
    # load model
    weight_path = f'weights/fold{current_fold_num}.pth'
    smartbind_model = load_smartbind_models(
        model_path=weight_path,
        device=device,
        vs_mode=True
        )
    # loop validation dataloader and perform inference
    rank_percentile_list = []
    smartbind_model.eval()
    with torch.no_grad():
        for batch in tqdm.tqdm(val_dataloader.dataloader):
            rna_sequences, match_smols, decoy_smols_list, _, _, _ = batch
            
            # Process RNA sequences
            rna_tokenized_list = []
            for rna_sequence in rna_sequences:
                rna_tokenized = smartbind_model.inference_single_rna(rna_sequence[1])
                rna_tokenized_list.append(rna_tokenized)
            
            # Process each RNA-ligand pair
            for anchor_rna, match_smol, decoy_smols in zip(
                rna_tokenized_list, match_smols, decoy_smols_list
            ):
                if len(decoy_smols) == 0:
                    print('No decoy smols for this pair, skip.')
                    continue
                
                # Process match smol - use inference_list_smols with single item
                anchor_rna = anchor_rna.squeeze(1)
                match_smol_tokenized = smartbind_model.inference_list_smols([match_smol[1]])[0]
                match_smol_tokenized = match_smol_tokenized.unsqueeze(dim=0)
                
                # Calculate positive distance
                match_distance = sigmoid_cosine_distance_test(anchor_rna, match_smol_tokenized)
    
                decoy_smol_tokenized_list = smartbind_model.inference_list_smols(decoy_smols)
                
                # Calculate distances
                decoy_distance_list = []
                for decoy_smol_tokenized in decoy_smol_tokenized_list:
                    decoy_smol_tokenized = decoy_smol_tokenized.unsqueeze(dim=0)
                    decoy_distance = sigmoid_cosine_distance_test(anchor_rna, decoy_smol_tokenized)
                    decoy_distance_list.append(decoy_distance)
                
                # Calculate rank percentile
                rank = 0
                for decoy_distance in decoy_distance_list:
                    if match_distance.item() <= decoy_distance.item():
                        rank += 1
                rank_percentile = (len(decoy_distance_list) + 1 - rank) / (len(decoy_distance_list) + 1)

                rank_percentile_list.append(rank_percentile)

    print(f'Inference completed! Total pairs: {len(rank_percentile_list)}')
    # Display rank percentile statistics
    mean_rank = np.mean(rank_percentile_list)
    median_rank = np.median(rank_percentile_list)
    print(f'Mean Rank Percentile: {mean_rank:.4f}')
    print(f'Median Rank Percentile: {median_rank:.4f}')
    full_rank_percentile_list.extend(rank_percentile_list)


Processing fold 1


Global seed set to 42


Not loaded: ['smol_binding_featurizer.0.weight', 'smol_binding_featurizer.0.bias', 'smol_binding_featurizer.1.weight', 'smol_binding_featurizer.1.bias', 'smol_binding_featurizer.1.running_mean', 'smol_binding_featurizer.1.running_var', 'smol_binding_featurizer.1.num_batches_tracked', 'binding_site_featurizer.cross_attention.smol_projection.weight', 'binding_site_featurizer.cross_attention.smol_projection.bias', 'binding_site_featurizer.cross_attention.projection.weight', 'binding_site_featurizer.cross_attention.projection.bias', 'binding_site_featurizer.cross_attention.batch_norm_projection.weight', 'binding_site_featurizer.cross_attention.batch_norm_projection.bias', 'binding_site_featurizer.cross_attention.batch_norm_projection.running_mean', 'binding_site_featurizer.cross_attention.batch_norm_projection.running_var', 'binding_site_featurizer.cross_attention.batch_norm_projection.num_batches_tracked', 'binding_site_featurizer.cross_attention.self_attention.0.0.in_proj_weight', 'bindi

100%|██████████| 3/3 [00:37<00:00, 12.62s/it]


Inference completed! Total pairs: 72
Mean Rank Percentile: 0.8341
Median Rank Percentile: 0.9149

Processing fold 2


...

Inference completed! Total pairs: 77
Mean Rank Percentile: 0.7848
Median Rank Percentile: 0.9375

Processing fold 10


Global seed set to 42


Not loaded: ['smol_binding_featurizer.0.weight', 'smol_binding_featurizer.0.bias', 'smol_binding_featurizer.1.weight', 'smol_binding_featurizer.1.bias', 'smol_binding_featurizer.1.running_mean', 'smol_binding_featurizer.1.running_var', 'smol_binding_featurizer.1.num_batches_tracked', 'binding_site_featurizer.cross_attention.smol_projection.weight', 'binding_site_featurizer.cross_attention.smol_projection.bias', 'binding_site_featurizer.cross_attention.projection.weight', 'binding_site_featurizer.cross_attention.projection.bias', 'binding_site_featurizer.cross_attention.batch_norm_projection.weight', 'binding_site_featurizer.cross_attention.batch_norm_projection.bias', 'binding_site_featurizer.cross_attention.batch_norm_projection.running_mean', 'binding_site_featurizer.cross_attention.batch_norm_projection.running_var', 'binding_site_featurizer.cross_attention.batch_norm_projection.num_batches_tracked', 'binding_site_featurizer.cross_attention.self_attention.0.0.in_proj_weight', 'bindi

100%|██████████| 4/4 [00:49<00:00, 12.50s/it]

Inference completed! Total pairs: 77
Mean Rank Percentile: 0.7769
Median Rank Percentile: 0.8113


In [21]:
print(f'Average Rank percentile is {np.mean(full_rank_percentile_list)}')
print('Standard deviation of Rank percentile is {:.4f}'.format(np.std(full_rank_percentile_list)))
# Average Rank percentile is 0.8104311804814506
# Standard deviation of Rank percentile is 0.2246

# Average Rank percentile is 0.8255608839971883
# Standard deviation of Rank percentile is 0.2254

Average Rank percentile is 0.8255608839971883
Standard deviation of Rank percentile is 0.2254


In [22]:
# 95% confidence interval
confidence_interval = 1.96 * np.std(full_rank_percentile_list) / np.sqrt(len(full_rank_percentile_list))
print(f'95% Confidence Interval: ±{confidence_interval:.4f}')

95% Confidence Interval: ±0.0160


In [23]:
# save 
import pickle
with open('plot_data/rnamigos1_pdbligand_rank_percentiles.pkl', 'wb') as f:
    pickle.dump(full_rank_percentile_list, f)

## Evaluate RNAmigos1 dataset with DecoyFinder decoys

In [24]:
decoy_db_path = 'data/rnamigos1_decoyfinder/clean'
full_rank_percentile_list = []
for current_fold_num in range(1, 11):
    print(f'\nProcessing fold {current_fold_num}')
    val_rna_seq_list, val_match_smol_list, val_rna_name_list, val_match_smol_name_list, val_binding_index_list, \
        val_match_smol_fp_list, train_rna_seq_list, train_match_smol_list, train_rna_name_list, \
        train_match_smol_name_list, train_binding_index_list, train_match_smol_fp_list = (
                                        build_val_test_set(
                                            fold_num=current_fold_num-1,
                                            dict_path=current_path,
                                            topk_decoy=topk_decoy))
    val_dataset = RLDataset(rna_sequences=val_rna_seq_list,
                                rna_sequences_names=val_rna_name_list,
                                match_smols=val_match_smol_fp_list,
                                match_smols_names=val_match_smol_name_list,
                                rna_smol_map_path='../../../../smartbind/rnamigos1_rna_smol_map.json',
                                non_binding_index_list=val_binding_index_list,
                                is_val=True)
    val_dataloader = RLDataLoader(dataset=val_dataset,
                                    batch_size=batch_size,
                                    num_workers=num_workers,
                                    if_shuffle=False)
    # load model
    weight_path = f'weights/fold{current_fold_num}.pth'
    smartbind_model = load_smartbind_models(
        model_path=weight_path,
        device=device,
        vs_mode=True
        )
    # loop validation dataloader and perform inference
    rank_percentile_list = []
    smartbind_model.eval()
    with torch.no_grad():
        for batch in tqdm.tqdm(val_dataloader.dataloader):
            rna_sequences, match_smols, _, _, _, _ = batch
            # build decoy smols list from decoy finder database
            decoy_smols_df_list = []
            for match_smol in match_smols:
                match_smol_name = match_smol[0]
                decoy_smols = []
                decoy_file_path = os.path.join(decoy_db_path, f'{match_smol_name}.txt')
                if os.path.exists(decoy_file_path):
                    with open(decoy_file_path, 'r') as f:
                        for line in f:
                            smol = line.strip()
                            pf2 = convert_smiles_to_pf2(smol)
                            decoy_smols.append(pf2)
                decoy_smols_df_list.append(decoy_smols)

            # Process RNA sequences
            rna_tokenized_list = []
            for rna_sequence in rna_sequences:
                rna_tokenized = smartbind_model.inference_single_rna(rna_sequence[1])
                rna_tokenized_list.append(rna_tokenized)
            
            # Process each RNA-ligand pair
            for anchor_rna, match_smol, decoy_smols in zip(
                rna_tokenized_list, match_smols, decoy_smols_df_list
            ):
                if len(decoy_smols) == 0:
                    print('No decoy smols for this pair, skip.')
                    continue
                
                # Process match smol - use inference_list_smols with single item
                anchor_rna = anchor_rna.squeeze(1)
                match_smol_tokenized = smartbind_model.inference_list_smols([match_smol[1]])[0]
                match_smol_tokenized = match_smol_tokenized.unsqueeze(dim=0)
                
                # Calculate positive distance
                match_distance = sigmoid_cosine_distance_test(anchor_rna, match_smol_tokenized)
                
                # Process decoy smols - batch process for efficiency
                decoy_smol_tokenized_list = smartbind_model.inference_list_smols(decoy_smols)
                
                # Calculate distances
                decoy_distance_list = []
                for decoy_smol_tokenized in decoy_smol_tokenized_list:
                    decoy_smol_tokenized = decoy_smol_tokenized.unsqueeze(dim=0)
                    decoy_distance = sigmoid_cosine_distance_test(anchor_rna, decoy_smol_tokenized)
                    decoy_distance_list.append(decoy_distance)
                
                # Calculate rank percentile
                rank = 0
                for decoy_distance in decoy_distance_list:
                    if match_distance.item() <= decoy_distance.item():
                        rank += 1
                rank_percentile = (len(decoy_distance_list) + 1 - rank) / (len(decoy_distance_list) + 1)

                rank_percentile_list.append(rank_percentile)

    print(f'Inference completed! Total pairs: {len(rank_percentile_list)}')
    # Display rank percentile statistics
    mean_rank = np.mean(rank_percentile_list)
    median_rank = np.median(rank_percentile_list)
    print(f'Mean Rank Percentile: {mean_rank:.4f}')
    print(f'Median Rank Percentile: {median_rank:.4f}')
    full_rank_percentile_list.extend(rank_percentile_list)


Processing fold 1


Global seed set to 42


Not loaded: ['smol_binding_featurizer.0.weight', 'smol_binding_featurizer.0.bias', 'smol_binding_featurizer.1.weight', 'smol_binding_featurizer.1.bias', 'smol_binding_featurizer.1.running_mean', 'smol_binding_featurizer.1.running_var', 'smol_binding_featurizer.1.num_batches_tracked', 'binding_site_featurizer.cross_attention.smol_projection.weight', 'binding_site_featurizer.cross_attention.smol_projection.bias', 'binding_site_featurizer.cross_attention.projection.weight', 'binding_site_featurizer.cross_attention.projection.bias', 'binding_site_featurizer.cross_attention.batch_norm_projection.weight', 'binding_site_featurizer.cross_attention.batch_norm_projection.bias', 'binding_site_featurizer.cross_attention.batch_norm_projection.running_mean', 'binding_site_featurizer.cross_attention.batch_norm_projection.running_var', 'binding_site_featurizer.cross_attention.batch_norm_projection.num_batches_tracked', 'binding_site_featurizer.cross_attention.self_attention.0.0.in_proj_weight', 'bindi

 33%|███▎      | 1/3 [00:12<00:24, 12.46s/it]

No decoy smols for this pair, skip.


...

Not loaded: ['smol_binding_featurizer.0.weight', 'smol_binding_featurizer.0.bias', 'smol_binding_featurizer.1.weight', 'smol_binding_featurizer.1.bias', 'smol_binding_featurizer.1.running_mean', 'smol_binding_featurizer.1.running_var', 'smol_binding_featurizer.1.num_batches_tracked', 'binding_site_featurizer.cross_attention.smol_projection.weight', 'binding_site_featurizer.cross_attention.smol_projection.bias', 'binding_site_featurizer.cross_attention.projection.weight', 'binding_site_featurizer.cross_attention.projection.bias', 'binding_site_featurizer.cross_attention.batch_norm_projection.weight', 'binding_site_featurizer.cross_attention.batch_norm_projection.bias', 'binding_site_featurizer.cross_attention.batch_norm_projection.running_mean', 'binding_site_featurizer.cross_attention.batch_norm_projection.running_var', 'binding_site_featurizer.cross_attention.batch_norm_projection.num_batches_tracked', 'binding_site_featurizer.cross_attention.self_attention.0.0.in_proj_weight', 'bindi

 75%|███████▌  | 3/4 [00:57<00:18, 18.68s/it]

No decoy smols for this pair, skip.
No decoy smols for this pair, skip.


100%|██████████| 4/4 [01:02<00:00, 15.66s/it]

Inference completed! Total pairs: 75
Mean Rank Percentile: 0.7458
Median Rank Percentile: 0.8571


In [25]:
print(f'Average Rank percentile is {np.mean(full_rank_percentile_list)}')
print('Standard deviation of Rank percentile is {:.4f}'.format(np.std(full_rank_percentile_list)))
# Average Rank percentile is 0.8794602376969063
# Standard deviation of Rank percentile is 0.2039

Average Rank percentile is 0.8794602376969063
Standard deviation of Rank percentile is 0.2039


In [26]:
confidence_interval = 1.96 * np.std(full_rank_percentile_list) / np.sqrt(len(full_rank_percentile_list))
print(f'95% Confidence Interval: ±{confidence_interval:.4f}')

95% Confidence Interval: ±0.0149


In [27]:
# save 
import pickle
with open('plot_data/rnamigos1_decoyfinder_rank_percentiles.pkl', 'wb') as f:
    pickle.dump(full_rank_percentile_list, f)